# SBL precursors at SuperKEKB – thesis analysis

This notebook reproduces the results of the master's thesis in the order of the chapters.
All logic lives in the modules in `files/`; the notebook only calls them.

**Hints**
- Run from top to bottom. Section 1 must always run first.
- The data (`Dataset/`) are **not** part of the repository (KEK / Belle II data).
- Figures are saved to `MasterArbeit/figures/` only where `save=...` is given.
- Clear all outputs before committing (`Kernel → Restart & Clear Output`).

## 1. Setup and data  

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt

DATASET = "Dataset"                                  # folder with the five CSV blocks
SBL_FILE = "Dataset/SBL_timestamps_Germany.csv"     # 128 SBL times, German local time
FIG = "MasterArbeit/figures"                         # output folder for thesis figures

### 00 EDA

In [ ]:
from files.data_loading import load_all_blocks, load_sbl_timestamps
from files.snapshot_analysis import load_raw_blocks

blocks = load_all_blocks(DATASET)        # forward-filled blocks (one row per snapshot, no NaN)
raw = load_raw_blocks(DATASET)           # raw blocks: a cell is set only when the channel reported
sbl_times = load_sbl_timestamps(SBL_FILE)
print({k: v.shape for k, v in blocks.items()}, len(sbl_times), "SBLs")

In [ ]:
from files.eda_plots import (
        plot_timeseries, plot_timeseries_multi_block,
        plot_feature_distributions, plot_missing_value_overview,
        plot_sbl_events_per_day,
    ) 
plot_timeseries_multi_block(blocks, log_scale=True, sbl_datetimes=sbl_times["timestamp"], title_prefix="Temporal Distribution of All Datasets and SBL Events Excluding Winter Shutdown Periods ", n_cols=2, y_zoom=False, y_quantile_low=0.01, y_quantile_high=0.99) 

In [ ]:
from files.eda_plots import (
    plot_timeseries, plot_timeseries_multi_block,
    plot_feature_distributions, plot_missing_value_overview,
    plot_sbl_events_per_day, plot_sbl_centered,
    plot_sbl_overlay, plot_sbl_summary,
)

In [ ]:
## filter name of the beam current channels
## 
filtered_beam = blocks["beam"].filter(regex="ler_current|her_current|timestamp")

plot_timeseries(
    filtered_beam,
    start_date=None,
    end_date=None,
    title_prefix="LER und HER Current",
    n_cols=1,
    sbl_datetimes=None,
    y_zoom=True,
    y_margin=0.05,
    y_quantile_low=0.01,
    y_quantile_high=0.99,
)
plot_timeseries(filtered_beam, start_date=None, end_date=None, title_prefix="LER und HER Current", n_cols=1,
                    sbl_datetimes=sbl_times['timestamp'], y_zoom=True, y_margin=0.05,
                    y_quantile_low=0.01, y_quantile_high=0.99)

In [ ]:
from files.current_around_sbl import (current_around_sbl, summarize_current_events,
                                check_time_offset, plot_current_around_sbl)

beam = blocks["beam"]

tab = current_around_sbl(
    beam, "timestamp", sbl_times,
    her_col="her_current",
    ler_col="ler_current",
    thr=1000,
)

summarize_current_events(tab)
check_time_offset(beam, "timestamp", sbl_times)
plot_current_around_sbl(
    beam, "timestamp", sbl_times,
    before="30min",
    after="60min",
)

In [ ]:
from  files.feature_distributions import plot_distribution_figures
plot_distribution_figures(
    blocks,
    distance_threshold=300,
    sbl_times=sbl_times["timestamp"],
    save_dir="Abbildung",
)

### 1.1 Shutdown and operating state
The winter shutdown is removed by hand; the beam state (BEAM_ON / BEAM_OFF / SHUTDOWN) comes from the
two beam currents (threshold 400 mA).

In [ ]:
from files.remove_shutdown import remove_time_ranges
from files.operation_state import operation_intervals, summary, plot_states

SHUTDOWN = [("2025-12-22", "2026-01-26")]
blocks = {k: remove_time_ranges(v, SHUTDOWN) for k, v in blocks.items()}

iv = operation_intervals(blocks["beam"], on_thr=400)
print(summary(iv, sbl_times["timestamp"]))
plot_states(blocks["beam"], iv)

In [ ]:
from files.beam_state_analysis import analyze_sbl_by_beam_state

state = analyze_sbl_by_beam_state(blocks, sbl_times)   # beam state of each ring at every SBL
state["summary"]

In [ ]:
res = analyze_sbl_by_beam_state(
    blocks,
    sbl_times,
    beam_block="beam",
    max_age="2h",
    max_gap="2h",
    her_thr=400,
    ler_thr=400,
    on_rule="any",
)

### 1.2 Sampling structure
The slow-control data are mainly 2-h snapshots. This check shows how many SBLs have a row
shortly before the event, and that the clocks of the blocks and of the SBL list agree.

In [ ]:
from files.snapshot_analysis import sampling_summary, plot_sampling, clock_check, plot_clock_check

display(sampling_summary(blocks, sbl_times))
plot_sampling(blocks, sbl_times)
plot_clock_check(clock_check(blocks, sbl_times))

## 2. Change frequency before an SBL  

In [ ]:
from files.p_before_sbl import plot_p_multi_window

# P per minute for several windows; compare with the chance level P0 = 1 min / T
fig, p_tab = plot_p_multi_window(sbl_times, blocks, windows=(30, 60, 120, 300, 1800), per_minute=True, top=20)

In [ ]:
from files.enrichment_analysis import enrichment_vs_buffer
from files.plot_buffer import plot_enrichment_vs_buffer

# Leave out the last 0-60 s before the SBL: a real precursor keeps its ratio, a loss response disappears
buf = enrichment_vs_buffer(sbl_times, blocks, window_seconds=300, buffers=(0, 5, 10, 30, 60))
plot_enrichment_vs_buffer(buf, top=10)    # save_path=f"{FIG}/r_buffer.png"

## 3. Last change before an SBL  

In [ ]:
import os
import matplotlib.pyplot as plt

# Zielordner erstellen, falls er noch nicht existiert
fe_art= 'Vacuum' ## Feature Art

# 1. Feature-Statistiken Plot aufrufen
from files.last_change_before_sbl import (
    last_feature_change_before_sbl,
    plot_feature_sbl_stats,
    plot_sbl_distance_histogram,
)

os.makedirs("Abbildung", exist_ok=True)# Berechnung der letzten Änderungen vor jedem SBL

block_name = fe_art.lower()
feature_data = blocks[block_name]
features = feature_data.select_dtypes("number").columns.tolist()

changes = last_feature_change_before_sbl(
    feature_data,
    "timestamp",
    features,
    sbl_times["timestamp"],
)
    



stats_df, (fig_features, ax_feat1, ax_feat2) = plot_feature_sbl_stats(
    changes,
    title=fe_art + "__Frequency and average for Last changed Feature before SBL per feature",
    figsize=(12, 5),
    show_plot=False,
)

os.makedirs("Abbildung", exist_ok=True)

stats_hist, (fig_hist, ax_hist) = plot_sbl_distance_histogram(
    changes_df=changes,
    title=fe_art + "__Last Change before SBL (Histogram)",
    distance_col="last_changed_distance_sec",
    figsize=(12, 5),
    show_plot=False,
)



# Beide Grafiken anzeigen
plt.show()

In [ ]:
from files.last_change_before_sbl import last_feature_change_before_sbl, plot_sbl_distance_histogram

for name, df in blocks.items():
    feats = [c for c in df.select_dtypes("number").columns]
    last = last_feature_change_before_sbl(df, "timestamp", feats, sbl_times["timestamp"])
    plot_sbl_distance_histogram(last, title=f"Last change before SBL ({name})")

## 4. Size of the changes: ECDF and KS test
Reference = beam-on rows more than 2 h away from any SBL.

In [ ]:
from files.ks_overview import ks_overview, plot_ks_overview
from files.ecdf_plot import plot_ecdf_window_vs_ref

ks = ks_overview(blocks, sbl_times, iv=iv)
plot_ks_overview(ks)                                   # save_as=f"{FIG}/ks_overview.png"
plot_ecdf_window_vs_ref(blocks, "timestamp", ["her_oscillator_freq", "ler_vac_pres_global"],
                        sbl_times, time_before=["5min", "30min"], iv=iv)

## 5. Isolation Forest 
One model per block, trained on normal operation. Takes a few minutes per window.
Images and tables are written to the output folders `sbl_iforest_*/` (ignored by git).

In [ ]:
from files.sbl_iforest import analyse_all, plot_feature_counts, plot_aggregate

res_120 = analyse_all(blocks, sbl_times, PRE_SBL_SECONDS=120, NORMAL_MARGIN_SECONDS=900,
                      OUTPUT_DIR="sbl_iforest_120s")
res_2h = analyse_all(blocks, sbl_times, PRE_SBL_SECONDS=7200, NORMAL_MARGIN_SECONDS=10800,
                     OUTPUT_DIR="sbl_iforest_2h")

In [ ]:
# Count per feature vs. chance level (black line); dark bars = clearly above chance
plot_feature_counts(res_2h, top_n=10)
plot_aggregate(res_2h, stat="median", top_per_block=5)

## 6. Supervised risk model  

In [ ]:
from files.risk_model import build_table, evaluate, plot_curves, plot_importance, plot_risk_time

tab = build_table(blocks, sbl_times, raw_blocks=raw)   # one row per 2-h snapshot with beam on
res = evaluate(tab)                                     # leave-one-month-out + chronological split
display(res["summary"])
plot_curves(res, split="chronological")
plot_importance(res)
plot_risk_time(res, sbl_times)

## 7. Discussion figures  

In [ ]:
from files.discussion_plots import time_since_last_row, plot_time_scales, plot_detectable_rate
from files.snapshot_analysis import activity_profile, plot_activity_profile

plot_time_scales(time_since_last_row(blocks, sbl_times))   # snapshot age vs. precursor time scale
plot_detectable_rate()                                     # how many events a feature needs above chance
prof = activity_profile(raw, blocks, sbl_times,
                        [("background", "her_oscillator_freq"), ("background", "ler_oscillator_freq")])
plot_activity_profile(prof)